# 02_EDA on Supply Chain Data to see what should be cleaned

In [0]:
%python

# Check raw data
df = spark.sql("SELECT * FROM supply_chain_demo.bronze.raw_supply_chain")

display(df)

- Want to change to snakcase

In [0]:
%python

# Check metadata df
df_metadata = spark.sql("FROM supply_chain_demo.bronze.metadata")
df_metadata.display()
     

In [0]:
%python
# COunt columns

df_metadata.count(), len(df.columns)

In [0]:
print(df.columns)

In [0]:
# Column names
df_metadata.select("FIELDS").display()

In [0]:
set(df_metadata.select("FIELDS").toPandas()["FIELDS"]).symmetric_difference(
    set(df.columns)
)

In [0]:
# count num of zipcodes
df.select("Order Zipcode").distinct().count()
     

In [0]:

print(f"Number of rows {df.count()}")
print(f"Number of columns {len(df.columns)}")

## Explore for cleaning columns

In [0]:
df.printSchema()

In [0]:
df.display()

In [0]:
df.select("Type").distinct().display()

In [0]:
df.select(
    [
        column
        for column, type_ in df.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()
     

In [0]:
df.select("Customer Country").distinct().display()

df.select("Order Country").distinct().display()

df.select("Customer Password").distinct().display()
df.select("Customer Zipcode").distinct().display()

In [0]:
df.select("Customer Zipcode").filter(df["Customer Zipcode"].isNull()).count()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
df.select("shipping date (DateOrders)").display()

In [0]:

df.select("Product Description").distinct().display()
     

In [0]:
display(df)

# Exercise
## Summary of cleaning steps
We note down what to be cleaned based on looking at the data, reading metadata and doing some EDA. This list is not comprehensive, I don't want to take away all the fun for you :)

## Drop
Customer Email (data is masked)
Customer Password
Product Description (all are nulls)

## Rounding errors
Order Item Product Price -> 2 decimals
Order Item Product Price -> 2 decimals
Benefit per order -> 2 decimals
Sales per customer -> 2 decimals ...

## Customer country
EE. UU. -> United States as it is the spanish abbreviation

## Nulls
fill Customer Lname with missing
fill Customer Zipcode with missing
fill Order Zipcode with missing

## Data type
shipping date - string -> Timestamp

## Derived columns 
will not remove them in silver layer, but will compute them in Gold layer instead

## Order Item Total 
derived from (Order Item Quantity)(Order Item Product Price)(1-Order Item Discount Rate), calculate and test this ...

## Rename columns
use snake_case convention instead (this is a design choice)

In [0]:

import re 

def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

df_column_alias = rename_columns_to_snake_case(df)

df_column_alias.limit(5).display()

In [0]:

from pyspark.sql.functions import to_timestamp, col 

df_timestamp = df_column_alias.withColumn(
    "shipping_date", to_timestamp(col("shipping_date_(dateorders)"), "M/d/yyyy H:mm")
)

df_timestamp.select("shipping_date").limit(3).display()

In [0]:
from pyspark.sql.functions import coalesce, lit, when

df_cleaned = (
    df_timestamp.withColumn("customer_lname", coalesce("customer_lname", lit("-")))
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
    )
    .withColumn(
        "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
    )
    .withColumn(
        "customer_country",
        when(col("customer_country") == "EE. UU.", "United States").otherwise(
            col("customer_country")
        ),
    )
).drop("product_description", "customer_email", "customer_password")


# display(df_cleaned.select("customer_country","customer_zipcode").distinct())
display(df_cleaned)

In [0]:
df_cleaned.display()